# Margin Risk Analysis — KINZ Products

**Goal:** Identify which KINZ products are most at risk of margin erosion based on historical competitor pricing data.

**Data sources:**
- `products` table: product catalog with COGS and alert thresholds
- `daily_prices` table: simulated competitor pricing (daily)
- `margin_history` table: calculated B2B + B2C margins per product per day
- `alerts` table: log of all triggered margin alerts

**Key questions:**
1. Which products have the lowest average margins?
2. Which products trigger the most alerts?
3. Is there a seasonal pattern to margin erosion?
4. Which products would benefit most from COGS renegotiation?

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from sqlalchemy import create_engine, text

try:
    fm.fontManager.addfont('/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf')
except Exception:
    pass
plt.rcParams['font.sans-serif'] = ['DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

from src.config import DATABASE_URL

engine = create_engine(DATABASE_URL)

# Load data
products = pd.read_sql(text('SELECT * FROM products WHERE active = true'), engine)
margins = pd.read_sql(text('SELECT mh.*, p.name, p.category FROM margin_history mh JOIN products p ON mh.product_id = p.id'), engine)
alerts = pd.read_sql(text('SELECT a.*, p.name FROM alerts a JOIN products p ON a.product_id = p.id'), engine)

print(f'Products: {len(products)}')
print(f'Margin records: {len(margins)}')
print(f'Alerts: {len(alerts)}')

## 1. Average Margins by Product (Risk Ranking)

In [ ]:
if not margins.empty:
    avg_margins = margins.groupby(['product_id', 'name', 'category']).agg(
        avg_b2c_margin=('b2c_margin_pct', 'mean'),
        avg_b2b_margin=('b2b_margin_pct', 'mean'),
        min_b2c_margin=('b2c_margin_pct', 'min'),
        min_b2b_margin=('b2b_margin_pct', 'min'),
        std_b2c_margin=('b2c_margin_pct', 'std'),
    ).reset_index().sort_values('avg_b2c_margin')
    
    print('=== Products ranked by average B2C margin (lowest = most at risk) ===')
    print(avg_margins[['name', 'category', 'avg_b2c_margin', 'avg_b2b_margin', 'min_b2c_margin']].to_string(index=False))
else:
    print('No margin data available. Run the pipeline first.')

## 2. Alert Frequency by Product

In [ ]:
if not alerts.empty:
    alert_counts = alerts.groupby(['name', 'alert_type']).size().unstack(fill_value=0)
    alert_counts['total'] = alert_counts.sum(axis=1)
    alert_counts = alert_counts.sort_values('total', ascending=False)
    
    print('=== Alert frequency by product (most alerts = highest risk) ===')
    print(alert_counts.to_string())
else:
    print('No alerts triggered — all margins are above threshold.')

## 3. Margin Volatility Analysis

Products with high margin volatility (std dev) are more likely to dip below threshold during price fluctuations.

In [ ]:
if not margins.empty:
    volatility = margins.groupby('name').agg(
        avg_margin=('b2c_margin_pct', 'mean'),
        std_margin=('b2c_margin_pct', 'std'),
        min_margin=('b2c_margin_pct', 'min'),
    ).sort_values('std_margin', ascending=False)
    
    print('=== Most volatile products (highest std dev = most unpredictable margins) ===')
    print(volatility.head(10).to_string())

## 4. Strategic Recommendations

Based on the analysis:

1. **High-risk products** (low avg margin + high volatility): prioritize COGS renegotiation or price increase
2. **Stable but low margin**: consider discontinuing or repricing
3. **High margin but volatile**: monitor closely — set tighter thresholds
4. **High margin + stable**: promote these products more aggressively

These recommendations should be validated with the KINZ procurement and marketing teams.